# Gameplay lifecycle stats

This notebook uses synthetic camera trajectories, so the analysis path needs no video decoder, model download, or GPU. Importing `kuration.gameplay` adds gameplay insight to the ordinary core `stats()` result.

In [ ]:
# Uncomment in a fresh JupyterLab environment:
# %pip install -e ".[stats]"

import math
import os
from pathlib import Path
from tempfile import TemporaryDirectory

import kuration.gameplay as gameplay
from kuration import SchemaSpec, Tick, input, stats

In [ ]:
schema = SchemaSpec.from_dict({
    "name": "notebook-gameplay",
    "entity_axis": "camera",
    "sample_hz": 10.0,
    "fields": {
        "camera_position": {
            "kind": "vector", "dims": ["x", "y", "z"], "semantics": "position"
        },
        "camera_z_axis": {"kind": "vector", "dims": ["x", "y", "z"]},
        "tracking_confidence": {"kind": "scalar"},
        "action": {"kind": "categorical"},
    },
})

def trajectory(speed, turn_rate, confidence):
    return [
        Tick(
            time=step / 10,
            entities={0: {
                "camera_position": (speed * step, math.sin(turn_rate * step), 0.0),
                "camera_z_axis": (
                    math.cos(turn_rate * step), math.sin(turn_rate * step), 0.0
                ),
                "tracking_confidence": confidence if step not in {14, 15} else confidence / 2,
                "action": "move" if step % 3 else "look",
            }},
        )
        for step in range(32)
    ]

sources = {
    "slow-pan": trajectory(0.03, 0.03, 0.98),
    "fast-turn": trajectory(0.12, 0.12, 0.90),
    "wide-arc": trajectory(0.07, 0.07, 0.95),
}

## Inspect candidates

`progress="auto"` updates compact progress rows in Jupyter and uses the terminal renderer in a script. The final expression renders as rich HTML.

In [ ]:
base = (
    input(sources, schema=schema)
    .window(8, stride=4)
    .slam_motion_diversity(axis="slam_turn_degrees", bins=4)
    .slam_quality(min_score=0.5)
    .options(progress="auto")
)

uncurated = base.stats()
uncurated

## Compare the curated set

The same report now includes candidate-to-selected changes. With gameplay imported, `plot()` automatically chooses the relevant domain chart; `available()` shows every supported plot and evidence mode.

In [ ]:
selected = base.sample("equal", n=6)
curated = selected.stats()
curated.plot()

## Inspect the evidence

Evidence ranking reuses the scored candidate pool. On a video-backed pipeline the same call adds bounded frame strips and action summaries.

In [ ]:
capabilities = curated.available()
print(capabilities)
evidence = curated.examples(
    metric="slam_quality_score", compare="selected-vs-rejected", n=3
)
evidence

## Inspect what was actually written

Artifact stats scan emitted arrays rather than reusing candidate estimates.

In [ ]:
workspace = TemporaryDirectory()
dataset_path = Path(workspace.name) / "curated"
written = selected.output(dataset_path, categorical="omit")
emitted = written.stats(progress="auto")
reopened_metadata = stats(dataset_path, content=False)
emitted

## Optional: compare gameplay videos

Set `KURATION_VIDEO_BEFORE` and `KURATION_VIDEO_AFTER` to show synchronized players. Small local videos embed automatically in notebook output; large files stay referenced to avoid bloating the notebook.

In [ ]:
before = os.environ.get("KURATION_VIDEO_BEFORE")
after = os.environ.get("KURATION_VIDEO_AFTER")
comparison = (
    gameplay.compare_videos(before, after, labels=("Before", "After"))
    if before and after
    else "Set both video environment variables to enable this cell."
)
comparison